# Part 7 — Is the Difference Real? Paired Significance Testing by Hand

*Evals from First Principles, Part 7.*

Part 6 built the eval set that scores your models. Now two models are graded on that same set: Model A gets 82%, Model B gets 84%. A 2-point win, or noise? Because both models were graded on the SAME items, the comparison is paired: we build the 2x2 of agreements and disagreements, run McNemar's test on only the discordant cells, derive the exact p-value by enumerating the binomial by hand, sanity-check it against the chi-square approximation, and finish with a power/sample-size back-of-envelope: how many items would you actually need to reliably catch a 2-point gain.

Pure Python + NumPy, offline, deterministic (no SciPy; the normal tail comes from `math.erfc` and the exact p-value from `math.comb`).

Companion script: `significance.py`

## The task

Two models answer the SAME 50 questions, graded 1 (correct) or 0 (wrong) per item. We construct the outcomes so Model A gets 41/50 (82%) and Model B gets 42/50 (84%), with the two models splitting on 9 items (the rest, both agree). A fixed seed (`numpy.random.default_rng(0)`) only shuffles which item sits at which index — it never changes the four cell counts below.

In [ ]:
import math
import numpy as np

# A fixed seed makes the shuffled item order reproducible. The shuffle only
# changes which item sits at which index; it never changes the four cell counts.
RNG = np.random.default_rng(0)


def build_outcomes():
    """Return two aligned 0/1 arrays: a[i], b[i] grade the SAME item i."""
    # groups: (both right) x37, (A only) x4, (B only) x5, (both wrong) x4
    a = np.array([1] * 37 + [1] * 4 + [0] * 5 + [0] * 4)
    b = np.array([1] * 37 + [0] * 4 + [1] * 5 + [0] * 4)
    order = RNG.permutation(len(a))          # interleave, like a real run
    return a[order], b[order]


a, b = build_outcomes()
n = len(a)
a_correct, b_correct = int(a.sum()), int(b.sum())
print(f"Two models graded on the SAME {n}-item set (1 = correct, 0 = wrong).")
print(f"  Model A: {a_correct}/{n} correct = {100*a_correct/n:.1f}%")
print(f"  Model B: {b_correct}/{n} correct = {100*b_correct/n:.1f}%")
print(f"  Headline gap: B - A = +{100*(b_correct-a_correct)/n:.1f} points. Real, or noise?")

disagree = [i for i in range(n) if a[i] != b[i]]
print(f"\nThe {len(disagree)} items where the models split (the rest, both agree):")
print("  item   A   B   who won")
for i in disagree:
    who = "A only" if a[i] == 1 else "B only"
    print(f"   {i:>3}   {a[i]}   {b[i]}   {who}")
# 9 discordant items: 5 go B's way, 4 go A's way

## Step 1 — The paired 2x2

Every item lands in exactly one of four cells: both models right, both wrong, or the two models split (A only, or B only). The two agreement cells (both-right, both-wrong) say nothing about which model is better — they contribute nothing to the comparison, since both models did the same thing on those items. All of the signal lives in the two DISCORDANT cells: `b` (A right, B wrong) and `c` (B right, A wrong).

In [ ]:
def paired_table(a, b):
    """The paired 2x2 counted by hand: (both_right, a_only, b_only, both_wrong)."""
    both_right = int(np.sum((a == 1) & (b == 1)))   # cell a
    a_only     = int(np.sum((a == 1) & (b == 0)))   # cell b: A right, B wrong
    b_only     = int(np.sum((a == 0) & (b == 1)))   # cell c: B right, A wrong
    both_wrong = int(np.sum((a == 0) & (b == 0)))   # cell d
    return both_right, a_only, b_only, both_wrong


both_right, a_only, b_only, both_wrong = paired_table(a, b)
print("                     B correct   B wrong")
print(f"  A correct              {both_right:>2}         {a_only:>2}      (a, b)")
print(f"  A wrong                 {b_only:>2}          {both_wrong:>2}      (c, d)")
print(f"  cells: a={both_right}  b={a_only}  c={b_only}  d={both_wrong}   (n={n})")
print(f"  Only the DISCORDANT cells carry the signal: b={a_only} (A right, B wrong)")
print(f"  and c={b_only} (B right, A wrong). The {both_right} both-right and {both_wrong} both-wrong")
print("  items say nothing about which model is better.")
# a=37  b=4  c=5  d=4

## Step 2 — McNemar's test on the discordant cells

Under the null hypothesis that the two models are equally good, each of the `b+c` disagreements is a coin flip: the count going B's way is `Binomial(b+c, 0.5)`. We compute the chi-square statistic two ways (uncorrected, and with the Yates continuity correction), then the exact two-sided p-value by summing binomial coefficients by hand: `2 * [C(9,0)+...+C(9,4)] / 2^9`. A 5-vs-4 split over 9 disagreements is exactly what a fair coin looks like.

In [ ]:
def mcnemar_chi2(b, c):
    """Chi-square statistics on the discordant cells (uncorrected and Yates)."""
    uncorrected = (b - c) ** 2 / (b + c)
    yates = (abs(b - c) - 1) ** 2 / (b + c)         # continuity correction
    return uncorrected, yates


def chi2_1df_pvalue(stat):
    """Two-sided p for a chi-square(df=1): P(X>stat) = erfc(sqrt(stat/2))."""
    return math.erfc(math.sqrt(stat / 2.0))


def mcnemar_exact_p(b, c):
    """
    Exact two-sided McNemar p-value, enumerated by hand. Under H0 the count of
    discordant pairs favoring one model ~ Binomial(n=b+c, 0.5). The two-sided p
    is 2 * P(X <= min(b,c)), capped at 1.0.
    """
    n = b + c
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1))   # C(n,0)+...+C(n,k)
    p = 2.0 * tail / (2 ** n)
    return min(1.0, p), n, k, tail


uncorr, yates = mcnemar_chi2(a_only, b_only)
p_chi = chi2_1df_pvalue(uncorr)
p_exact, disc_n, k, tail = mcnemar_exact_p(a_only, b_only)
print(f"  chi-square (uncorrected) = (b-c)^2/(b+c) = ({a_only}-{b_only})^2/{a_only+b_only} = {uncorr:.3f}")
print(f"  chi-square (Yates corr.) = (|b-c|-1)^2/(b+c) = 0^2/{a_only+b_only} = {yates:.3f}")
print(f"  approx p (chi-square, 1 df) = erfc(sqrt({uncorr:.3f}/2)) = {p_chi:.3f}")
print(f"  exact p: 2 * [C({disc_n},0)+...+C({disc_n},{k})] / 2^{disc_n}")
print(f"         = 2 * {tail} / {2**disc_n} = {p_exact:.3f}")
print(f"  -> p = {p_exact:.3f} >> 0.05. NOT significant: we cannot rule out noise.")
# chi-square uncorrected=0.111  Yates=0.000  approx p=0.739  exact p=1.000

## Step 3 — Power: how big a set to detect a TRUE 2-point gain?

A non-significant result on 50 items does not mean the models are equal, it may just mean the eval set was too small to see a real 2-point gap. We compute the sample size two ways: the standard unpaired two-proportion rule (treating the two models' scores as independent), and the paired McNemar rule (using the same coin-flip logic as the test itself, applied to the fraction of items that come out discordant). Both target `alpha=0.05` (`z=1.96`) and 80% power (`z=0.84`).

In [ ]:
# Standard normal critical values for alpha=0.05 two-sided and 80% power.
Z_ALPHA = 1.96      # P(|Z| > 1.96) = 0.05
Z_BETA = 0.84        # P(Z > 0.84)  = 0.20  -> 80% power


def sample_size_unpaired(pA, pB):
    """Two-proportion back-of-envelope: items PER MODEL to detect pB - pA."""
    var = pA * (1 - pA) + pB * (1 - pB)
    delta = pB - pA
    n = (Z_ALPHA + Z_BETA) ** 2 * var / delta ** 2
    return math.ceil(n)


def sample_size_paired(p_a_only, p_b_only):
    """
    Paired (McNemar) sample size, same coin-flip logic as the test itself.
    Discordant pairs are a coin with P(favor B) = rho; detecting rho vs 0.5 is a
    one-sample proportion test. Returns (discordant_pairs_needed, items_needed).
    """
    pi_d = p_a_only + p_b_only               # P(the two models disagree)
    rho = p_b_only / pi_d                     # among disagreements, share for B
    num = Z_ALPHA * math.sqrt(0.25) + Z_BETA * math.sqrt(rho * (1 - rho))
    m = num ** 2 / (rho - 0.5) ** 2           # discordant pairs needed
    return math.ceil(m), math.ceil(m / pi_d)


pA, pB = a_correct / n, b_correct / n
n_unpaired = sample_size_unpaired(pA, pB)
disc_needed, n_paired = sample_size_paired(a_only / n, b_only / n)
print(f"  Target: detect pB-pA = {pB-pA:.2f} at alpha=0.05 (z={Z_ALPHA}), 80% power (z={Z_BETA}).")
print("  Unpaired two-proportion rule:")
print("    n = (z_a+z_b)^2 * [pA(1-pA)+pB(1-pB)] / (pB-pA)^2")
print(f"      = {(Z_ALPHA+Z_BETA)**2:.2f} * {pA*(1-pA)+pB*(1-pB):.4f} / {(pB-pA)**2:.4f} = {n_unpaired} items PER MODEL.")
print(f"  Paired (McNemar) rule, same coin logic, {100*(a_only+b_only)/n:.0f}% of items discordant:")
print(f"    need {disc_needed} discordant pairs -> {n_paired} items total.")
print(f"  -> Thousands of items to catch a 2-point gap 80% of the time. Pairing")
print(f"     helps ({n_paired} < {n_unpaired}), but a 2-point headline on a 50-item eval is a vibe, not a result.")
# unpaired n=5528 per model, paired discordant needed=633, paired n=3515 items

## Recap

- A headline gap between two models is not automatically a real difference. When both models are graded on the SAME set, the comparison is paired, and only the DISCORDANT items (where the two models split) carry any signal.
- **McNemar's test** treats the discordant count as a coin flip under the null of equal models. Here a 5-vs-4 split over 9 disagreements gives an exact p-value of 1.000, indistinguishable from a fair coin: not significant.
- A non-significant result does not prove the models are equal, it can simply mean the eval set is too small. The power calculation showed we would need thousands of items, not 50, to reliably catch a true 2-point gain.
- **Pairing helps**: because it isolates the discordant items, the paired sample-size requirement (3515 items) is smaller than the unpaired one (5528 items) for the same target gap.

Next: what happens when the "gold" labels themselves disagree, and how to measure inter-annotator agreement.